# (IIP314W-2) [2026-T2] OPTIMIZACIÓN APLICADA A NEGOCIOS
## Ayudantía 8: Repaso integrador — modelamiento, dualidad y KKT

---

**Profesor:** Ing. Rodrigo Trigo Vilches

**Ayudante:** Lic. Vicente Ramírez

**Fecha:** 12 de agosto de 2026

**Universidad del Desarrollo**

---

## Objetivos

Esta es la **última ayudantía del curso**, y es deliberadamente **integradora**: recorre en un solo hilo todo lo que vimos, sobre un mismo negocio.

- Formular un modelo **entero-mixto** que combine **restricciones de todos los tipos**: igualdades de balance, requerimientos mínimos ($\ge$), disponibilidades ($\le$), un **indicador binario de dos lados** (lote mínimo + costo fijo) y cotas de demanda.
- Construir el **dual** de un LP cuyo primal tiene restricciones $\le$ **y** $\ge$, y leer correctamente los **signos** de las variables duales.
- Verificar **dualidad fuerte** y **holguras complementarias**, e interpretar cada precio sombra —incluido uno **negativo**— en términos de negocio.
- Escribir las **condiciones KKT** y comprobar que, para un LP, **KKT no es otra cosa que la dualidad**: la estacionariedad *es* la factibilidad dual, y la holgura complementaria *es* la misma de siempre. Los **multiplicadores $\mu_i$ son los precios sombra**.
- Resolver un problema **no lineal** por **KKT enumerando casos**, verificar la **concavidad** que garantiza que el punto es óptimo **global**, y comprobar que $\mu_i=\partial f^*/\partial b_i$ **también** en el caso no lineal.

> **Conexión con las clases.** En la **Clase 29 (Repaso)** el profesor cerró con **Lagrange** y **KKT** en programación no lineal: convexidad como condición de suficiencia, el Lagrangiano, y la resolución **revisando todas las combinaciones de restricciones activas e inactivas**. En la **Ayudantía 6** construimos el dual y leímos los precios sombra en el tableau; en la **Ayudantía 7** los usamos para análisis de sensibilidad. Esta ayudantía **cierra el círculo**: muestra que dualidad, precios sombra y holgura complementaria son **casos particulares de KKT**.

## Sección 0 — Repaso

### 0.1 Un catálogo de restricciones

Casi todo modelo de negocio se arma con cinco o seis piezas. Conviene tenerlas identificadas por **tipo**, porque el tipo determina el signo de su variable dual.

| Estructura | Forma | Tipo | Ejemplo |
|:--|:--|:--:|:--|
| **Balance de masa** | $p_i=\sum_k x_{ki}$ | $=$ | lo producido es la suma de los insumos que entraron |
| **Requerimiento de calidad** | $\sum_k \pi_k x_{ki}\;\ge\;\pi^{\min}_i\,p_i$ | $\ge$ | la mezcla debe tener al menos $38\%$ de proteína |
| **Disponibilidad de recurso** | $\sum_i x_{ki}\le S_k$ | $\le$ | no hay más de $150$ t de harina de pescado al mes |
| **Capacidad** | $\sum_i p_i\le C$ | $\le$ | la extrusora procesa $1.100$ t/mes |
| **Balance de inventario** | $r_{it}=r_{i,t-1}+p_{it}-v_{it}$ | $=$ | lo que queda es lo que había, más lo que entró, menos lo que salió |
| **Indicador de dos lados** | $L^{\min}y_i\le p_i\le M\,y_i$ | binaria | *"o no se produce, o se produce al menos un lote"* |
| **Cotas** | $\underline d_i\le v_i\le \bar d_i$ | $lb/ub$ | compromiso contractual y demanda máxima |

**Sobre el requerimiento de calidad.** La tentación es escribirlo como cociente,
$$\frac{\sum_k \pi_k x_{ki}}{\sum_k x_{ki}}\ge \pi^{\min}_i,$$
que **no es lineal** y está indefinido si no se produce nada. Se **multiplica cruzado** y queda lineal. Es la misma maniobra de las proporciones de mezcla de la Ayudantía 7.

**Sobre el indicador de dos lados.** Es la pieza nueva de hoy. Modela *"o cero, o al menos $L^{\min}$"* —una **disyunción**, que es intrínsecamente no convexa y por eso **necesita** una binaria:

$$\boxed{\;L^{\min}\,y_i\;\le\;p_i\;\le\;M\,y_i\;}\qquad y_i\in\{0,1\}$$

- $y_i=0\;\Rightarrow\;0\le p_i\le 0$: la línea está **apagada**.
- $y_i=1\;\Rightarrow\;L^{\min}\le p_i\le M$: si se enciende, hay que hacer **al menos un lote**.

El costo fijo $F\,y_i$ en la objetivo es lo que hace que apagar la línea sea a veces lo correcto. Y como siempre: **$M$ debe ser la cota válida más chica** (aquí, la capacidad de la extrusora), no $10^9$.

### 0.2 El dual cuando hay restricciones de varios tipos

La regla de la Ayudantía 6 servía para el caso "todas $\le$". La versión general para un **primal de maximización** es:

| Primal ($\max\;c^\top x$) | Dual ($\min\;b^\top y$) |
|:--|:--|
| restricción $i$ de tipo $\;\le b_i$ | $y_i\ \ge 0$ |
| restricción $i$ de tipo $\;\ge b_i$ | $y_i\ \le 0$ |
| restricción $i$ de tipo $\;= b_i$ | $y_i$ **libre** |
| variable $x_j\ge0$ | restricción dual $j$ de tipo $\ \ge c_j$ |
| variable $x_j$ libre | restricción dual $j$ de tipo $\ = c_j$ |

**La regla mnemotécnica y su sentido económico.** En un máximo, **relajar un techo nunca puede empeorar** el óptimo $\Rightarrow y_i\ge0$ para las $\le$. En cambio, **subir un piso nunca puede mejorarlo** $\Rightarrow y_i\le0$ para las $\ge$: un compromiso mínimo más exigente es una **obligación**, y su precio sombra mide **cuánto cuesta**, no cuánto vale.

Una restricción $\ge$ siempre se puede llevar a $\le$ multiplicando por $-1$ ($a_i^\top x\ge b_i\iff -a_i^\top x\le -b_i$); el multiplicador $\mu_i\ge0$ de la forma $\le$ y el precio sombra $y_i$ del problema original cumplen $y_i=-\mu_i$. **Es el mismo número con distinto signo**, y hay que ser explícito sobre cuál se está reportando. Gurobi reporta siempre $y_i=\partial z^*/\partial b_i$ sobre la restricción **tal como fue escrita**.

**Dualidad débil / fuerte y holguras complementarias** (igual que en la Ayudantía 6):

$$c^\top x\le b^\top y\ \ \forall\text{ par factible},\qquad c^\top x^*=b^\top y^*,\qquad
\underbrace{y_i^*\,\big(b_i-a_i^\top x^*\big)=0}_{\text{por restricción}},\qquad
\underbrace{x_j^*\,\big(a_j^\top y^*-c_j\big)=0}_{\text{por variable}}.$$

### 0.3 KKT: la teoría que contiene a todo lo anterior

Las condiciones de **Karush–Kuhn–Tucker** son las condiciones de optimalidad de **cualquier** problema con restricciones, lineal o no. Para un problema de **maximización** escrito con todas las restricciones en forma $g_i(x)\le0$:

$$\max_x\; f(x)\qquad\text{s.a.}\quad g_i(x)\le0\ \ (i=1,\dots,m)$$

el **Lagrangiano** es

$$\boxed{\;\mathcal L(x,\mu)=f(x)-\sum_{i=1}^m \mu_i\,g_i(x)\;}$$

y las condiciones KKT en $(x^*,\mu^*)$ son cuatro:

| # | Condición | Forma | Qué dice |
|:--:|:--|:--|:--|
| 1 | **Estacionariedad** | $\nabla f(x^*)=\displaystyle\sum_i \mu_i^*\,\nabla g_i(x^*)$ | el gradiente de la ganancia es una combinación de los gradientes de las restricciones que topan |
| 2 | **Factibilidad primal** | $g_i(x^*)\le0\ \ \forall i$ | la solución es factible |
| 3 | **Factibilidad dual** | $\mu_i^*\ge0\ \ \forall i$ | los multiplicadores no son negativos |
| 4 | **Holgura complementaria** | $\mu_i^*\,g_i(x^*)=0\ \ \forall i$ | o la restricción está **activa**, o su multiplicador es **cero** |

Y el resultado que da sentido económico a todo:

$$\boxed{\;\mu_i^*=\frac{\partial f^*}{\partial b_i}\;}$$

**el multiplicador de KKT *es* el precio sombra.** La condición 4 es **exactamente** la holgura complementaria de la Ayudantía 6: *si sobra recurso, no pago por más; si pago por más, es porque está agotado.*

> **Cuándo sirven.** KKT es **necesaria** bajo una **cualificación de restricciones** (basta que las restricciones sean lineales, o la condición de Slater: que exista un punto **estrictamente** factible). Es **suficiente** —y entrega el óptimo **global**— cuando el problema es **cóncavo** (maximizar $f$ cóncava sobre un conjunto convexo). Un LP cumple ambas cosas siempre.

**Cómo se resuelve a mano.** Como en la Clase 29: se **enumeran las combinaciones de restricciones activas e inactivas**. Para cada combinación se impone $\mu_i=0$ en las inactivas y $g_i(x)=0$ en las activas, se resuelve el sistema, y se **descarta** el caso si aparece $\mu_i<0$, si $x$ viola alguna restricción, o si el sistema es incompatible. Con $m$ restricciones son $2^m$ casos, pero la mayoría se cae en una línea.

### 0.4 KKT aplicado a un LP: **es la dualidad**

Tomemos el LP $\;\max\,c^\top x\;$ s.a. $\;Ax\le b,\;x\ge0$. Las restricciones son $g_i(x)=a_i^\top x-b_i\le0$ y $g^{neg}_j(x)=-x_j\le0$, con multiplicadores $\mu_i$ y $\sigma_j$. El Lagrangiano es

$$\mathcal L=c^\top x-\sum_i\mu_i\,(a_i^\top x-b_i)+\sum_j\sigma_j x_j,$$

y la estacionariedad respecto de $x_j$ da

$$c_j-\sum_i \mu_i a_{ij}+\sigma_j=0 \;\;\Longleftrightarrow\;\; \sum_i \mu_i a_{ij}=c_j+\sigma_j\;\ge\;c_j\quad(\text{pues }\sigma_j\ge0).$$

Es decir $A^\top\mu\ge c$ con $\mu\ge0$: **la factibilidad del dual**. Y la holgura complementaria de $\sigma_j$ dice $\sigma_j x_j=0$, o sea $x_j>0\Rightarrow\sigma_j=0\Rightarrow$ la restricción dual $j$ está **activa**. La tabla completa:

| KKT | Programación lineal |
|:--|:--|
| Estacionariedad | **Factibilidad dual**: $A^\top y\ge c$ |
| Factibilidad primal | $Ax\le b,\;x\ge0$ |
| $\mu_i\ge0$ | $y_i\ge0$ para las restricciones $\le$ |
| Holgura complementaria en $g_i$ | $y_i\,(b_i-a_i^\top x)=0$ |
| Holgura complementaria en $-x_j\le0$ | $x_j\,(a_j^\top y-c_j)=0$ |
| $\mu_i=\partial f^*/\partial b_i$ | **precio sombra** |

$$\boxed{\;\mu^*=y^*\;}$$

**Los multiplicadores de KKT y las variables duales son el mismo objeto.** Todo lo que hicimos con el tableau en la Ayudantía 6 era KKT sin decirlo. La diferencia es que **KKT sigue funcionando cuando el problema deja de ser lineal**, que es lo que veremos en el Ejercicio 3.

---

# Ejercicio 1 — Planta de alimento para salmones "AquaNutrí Chiloé"

### Contexto

**AquaNutrí Chiloé** fabrica alimento extruido para salmonicultura y debe planificar los próximos **tres meses**. Produce dos formulaciones:

- **Engorda** (EN), el producto de volumen, que va a las balsas-jaula de engorda;
- **Smolt** (SM), un alimento de alta proteína para peces jóvenes en agua dulce, más caro y más rentable.

Ambos se fabrican mezclando tres insumos: **harina de pescado** (HP), **concentrado de soya** (SOY) y **harina de trigo** (TRI). Cada insumo tiene un **contenido proteico** $\pi_k$ y un costo $c_k$ por tonelada; cada formulación exige un **mínimo de proteína** $\pi^{\min}_i$. La harina de pescado es la única con proteína suficientemente alta para el Smolt, y es también la **escasa**: hay un cupo mensual $S_{\text{HP},t}$ por la cuota de pesca.

La **extrusora** es una sola línea y procesa hasta $C$ toneladas al mes, sumando ambos productos. Cambiar de formulación exige **detener, limpiar y recalibrar** la línea: por eso, producir un producto en un mes tiene un **costo fijo de preparación** $F$, y si se produce, se produce **al menos un lote de $L^{\min}$ toneladas** (por debajo de eso la corrida no es estable).

Lo producido y no vendido queda en **bodega** (capacidad $R$ toneladas, costo $h$ por tonelada y mes). Cada mes hay un **compromiso mínimo** $\underline d_{it}$ con los clientes y una **demanda máxima** $\bar d_{it}$ que el mercado absorbe.

**Objetivo:** maximizar el margen de los tres meses (ingresos − insumos − preparaciones − almacenamiento).

### Conjuntos y parámetros

| Símbolo | Descripción |
|:--|:--|
| $\mathcal K=\{\text{HP},\text{SOY},\text{TRI}\}$ | insumos (índice $k$) |
| $\mathcal I=\{\text{EN},\text{SM}\}$ | formulaciones (índice $i$) |
| $\mathcal T=\{1,2,3\}$ | meses (índice $t$) |

| Símbolo | Significado | Unidad |
|:--|:--|:--|
| $\pi_k$ | contenido proteico del insumo $k$ | fracción |
| $c_k$ | costo del insumo $k$ | \$ mil/t |
| $\pi^{\min}_i$ | proteína mínima exigida a la formulación $i$ | fracción |
| $pr_i$ | precio de venta de la formulación $i$ | \$ mil/t |
| $S_{kt}$ | disponibilidad del insumo $k$ en el mes $t$ | t |
| $C$ | capacidad de la extrusora | t/mes |
| $L^{\min}$ | lote mínimo de una corrida | t |
| $F$ | costo fijo de preparación de una corrida | \$ mil |
| $h$ | costo de almacenamiento | \$ mil/(t·mes) |
| $R$ | capacidad de bodega | t |
| $\underline d_{it},\ \bar d_{it}$ | compromiso mínimo y demanda máxima | t |
| $r^0_i$ | inventario inicial | t |

### Datos de la instancia

| Insumo | $\pi_k$ (proteína) | $c_k$ (\$ mil/t) | | Formulación | $\pi^{\min}_i$ | $pr_i$ (\$ mil/t) |
|:--|--:|--:|:--|:--|--:|--:|
| Harina de pescado (HP) | 0,65 | 1.400 | | Engorda (EN) | 0,38 | 1.150 |
| Concentrado de soya (SOY) | 0,45 | 700 | | Smolt (SM) | 0,55 | 1.600 |
| Harina de trigo (TRI) | 0,10 | 300 | | | | |

| Escalar | $C$ | $L^{\min}$ | $F$ | $h$ | $R$ |
|:--|--:|--:|--:|--:|--:|
| valor | 1.100 | 150 | 60.000 | 15 | 400 |

| Disponibilidad $S_{kt}$ (t) | $t=1$ | $t=2$ | $t=3$ | | Demanda (t) | $t=1$ | $t=2$ | $t=3$ |
|:--|--:|--:|--:|:--|:--|--:|--:|--:|
| HP | 150 | 150 | 150 | | $\bar d_{\text{EN},t}$ | 700 | 900 | 1.100 |
| SOY | 900 | 900 | 900 | | $\bar d_{\text{SM},t}$ | 120 | 320 | 400 |
| TRI | 400 | 400 | 400 | | $\underline d_{\text{EN},t}$ | 300 | 400 | 500 |
| | | | | | $\underline d_{\text{SM},t}$ | 0 | 0 | 0 |

Inventario inicial $r^0_i=0$.

### Se pide

1. Defina **variables de decisión** con su dominio, **función objetivo** y las **seis familias de restricciones**, en notación de sumatorias con su $\forall$. Identifique el **tipo** de cada familia ($=$, $\ge$, $\le$, indicador binario, cota).
2. Justifique el valor de $M$ en el indicador de dos lados y explique qué modela cada uno de sus dos lados.
3. ¿Por qué el requerimiento de proteína **no** se escribe como cociente?
4. Implemente el modelo en `gurobipy` y resuélvalo. Reporte el margen óptimo, el plan de producción, venta e inventario, y en qué meses se enciende cada línea.
5. **Interprete el plan**: ¿cuál es el cuello de botella? ¿por qué se produce Smolt por adelantado? ¿por qué **no** se enciende la línea de Smolt en el mes 3, si es el producto de mayor margen? ¿queda demanda sin atender?
6. Verifique numéricamente que la solución cumple las seis familias de restricciones.

---

## 1.1 — Formulación (preguntas 1, 2 y 3)

_Plantee el modelo de forma **algebraica** (sumatorias, parámetros simbólicos, **sin números**): variables con su dominio, función objetivo y las **seis familias de restricciones** con su $\forall$, indicando el **tipo** de cada una. Responda también las preguntas 2 y 3._

## 1.4 — Resolución en Gurobi

Los datos se entregan en un **diccionario** y el modelo va en una **función**: así el mismo código sirve para el MIP y para los escenarios *what if*. **Ningún número debe quedar escrito dentro del modelo.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB, quicksum
from scipy.optimize import minimize, linprog
import itertools

### Los datos (se entregan)

In [ ]:
def datos():
    """Instancia: 3 insumos x 2 formulaciones x 3 meses. Precios y costos en $ mil por tonelada."""
    d = {}
    d["K"] = ["HP", "SOY", "TRI"]                 # insumos
    d["I"] = ["EN", "SM"]                         # formulaciones
    d["T"] = [1, 2, 3]                            # meses

    d["prot"] = {"HP": 0.65, "SOY": 0.45, "TRI": 0.10}      # contenido proteico
    d["c"]    = {"HP": 1400., "SOY": 700., "TRI": 300.}     # costo del insumo
    d["pmin"] = {"EN": 0.38, "SM": 0.55}                    # proteina minima exigida
    d["pr"]   = {"EN": 1150., "SM": 1600.}                  # precio de venta

    d["S"] = {("HP", t): 150. for t in [1, 2, 3]}           # disponibilidad mensual
    d["S"].update({("SOY", t): 900. for t in [1, 2, 3]})
    d["S"].update({("TRI", t): 400. for t in [1, 2, 3]})

    d["C"]    = 1100.        # capacidad de la extrusora (t/mes)  ->  tambien es la M valida mas chica
    d["Lmin"] = 150.         # lote minimo de una corrida (t)
    d["F"]    = 60000.       # costo fijo de preparacion de una corrida ($ mil)
    d["h"]    = 15.          # almacenamiento ($ mil / t-mes)
    d["R"]    = 400.         # capacidad de bodega (t)

    d["dmax"] = {("EN",1): 700., ("EN",2): 900., ("EN",3): 1100.,
                 ("SM",1): 120., ("SM",2): 320., ("SM",3):  400.}
    d["dmin"] = {("EN",1): 300., ("EN",2): 400., ("EN",3):  500.,
                 ("SM",1):   0., ("SM",2):   0., ("SM",3):    0.}
    d["r0"]   = {"EN": 0., "SM": 0.}
    return d


D = datos()
print("M = C =", D["C"], "t   (la cota valida mas chica para p_it)")
print("Capacidad total del horizonte:", 3*D["C"], "t")
print("Demanda maxima total        :", sum(D["dmax"].values()), "t"
      "   ->  no alcanza para todo")

### El modelo

_Complete la función. Se deja escrita la **primera** familia como ejemplo del estilo que se espera: cada familia con **nombre**, porque después se piden sus duales por nombre._

In [ ]:
def construir_modelo(d, ybar=None, **cambios):
    """MIP de AquaNutri Chiloe.

    ybar=None            -> MIP (y_it binaria).
    ybar={(i,t): 0/1}    -> LP puro: y entra como DATO.
    **cambios            -> sobrescribe parametros para escenarios "what if".
    """
    d = {**d, **cambios}
    K, I, T = d["K"], d["I"], d["T"]
    t0 = T[0]

    m = gp.Model("aquanutri")
    m.Params.OutputFlag = 0
    m.Params.MIPGap = 0.0                       # optimo exacto: los escenarios deben ser comparables

    x = m.addVars(K, I, T, name="x")                                 # insumo k -> formulacion i
    p = m.addVars(I, T, name="p")                                    # produccion
    v = m.addVars(I, T, lb=d["dmin"], ub=d["dmax"], name="v")        # ventas (con cotas)
    r = m.addVars(I, T, name="r")                                    # inventario

    if ybar is None:
        y = m.addVars(I, T, vtype=GRB.BINARY, name="y")
        costo_fijo = quicksum(d["F"]*y[i,t] for i in I for t in T)
    else:
        y = ybar                                                     # numeros, no variables
        costo_fijo = sum(d["F"]*y[i,t] for i in I for t in T)

    # ---- funcion objetivo: ingresos - insumos - preparaciones - almacenamiento
    #### CÓDIGO AQUÍ ####

    # (1) =   balance de masa   [EJEMPLO -- las demas van en el mismo estilo]
    m.addConstrs((p[i,t] == quicksum(x[k,i,t] for k in K)
                  for i in I for t in T), "masa")
    # (2) >=  proteina minima   (multiplicada en cruz: NO es un cociente)
    #### CÓDIGO AQUÍ ####
    # (3) <=  disponibilidad de cada insumo
    #### CÓDIGO AQUÍ ####
    # (4) <=  capacidad de la extrusora
    #### CÓDIGO AQUÍ ####
    # (5) binaria: indicador de DOS LADOS  ->  p = 0  o  p >= Lmin
    #     ojo con el valor de M
    #### CÓDIGO AQUÍ ####
    # (6) =   balance de inventario  (+ capacidad de bodega)
    #### CÓDIGO AQUÍ ####

    m._var = dict(x=x, p=p, v=v, r=r, y=y); m._d = d
    m.update()
    return m


def resolver(d, **cambios):
    m = construir_modelo(d, **cambios); m.optimize(); return m

### El plan óptimo (pregunta 4)

In [ ]:
m1 = resolver(D)
V, K, I, T = m1._var, D["K"], D["I"], D["T"]
ystar = {(i,t): int(round(V["y"][i,t].X)) for i in I for t in T}

print(f"Estado: {m1.Status} (2 = OPTIMAL)")
print(f"Margen optimo = {m1.ObjVal:,.0f}  ($ mil)")
print(f"{m1.NumVars} variables ({m1.NumBinVars} binarias)  |  {m1.NumConstrs} restricciones")

# Reporte el plan: lineas encendidas y*_it, produccion, venta, inventario,
# uso de cada insumo contra su disponibilidad, y la receta efectiva de cada formulacion.
# Identifique tambien que demanda queda sin atender.
#### CÓDIGO AQUÍ ####

_**Interprete el plan (pregunta 5).** ¿Cuál es el cuello de botella? ¿Por qué se produce Smolt por adelantado y se guarda inventario? ¿Por qué **no** se enciende la línea de Smolt en el mes 3, si es el producto de mayor margen —y si además sobra harina de pescado ese mes? Haga el cálculo del diferencial de margen contra el costo fijo de preparación. ¿Qué demanda queda sin atender y quién la está bloqueando?_

### Verificación de factibilidad (pregunta 6)

In [ ]:
# Verifique numericamente que la solucion cumple las SEIS familias de restricciones
# (mas la capacidad de bodega y las cotas de demanda). Compruebe ademas el margen
# unitario de cada formulacion a partir de la receta que eligio el modelo.
#### CÓDIGO AQUÍ ####

---

# Ejercicio 2 — El mes de régimen: **primal y dual**

El modelo del Ejercicio 1 es un **MIP**, y como vimos en la Ayudantía 7, **un MIP no tiene precios sombra**. Para poder hablar de dualidad nos quedamos con un **mes de régimen** y aprovechamos lo que ya aprendimos del Ejercicio 1: las **recetas óptimas son fijas** ($80/20$ para el Engorda, $50/50$ para el Smolt), así que cada formulación se puede describir por lo que **consume** y lo que **aporta**:

| Por tonelada producida | Engorda $x_1$ | Smolt $x_2$ | Disponible |
|:--|--:|--:|--:|
| Extrusora (t) | $1$ | $1$ | $\le 1.100$ |
| Harina de pescado (t) | $0$ | $0{,}5$ | $\le 150$ |
| Concentrado de soya (t) | $0{,}8$ | $0{,}5$ | $\le 900$ |
| **Margen (\$ mil/t)** | **530** | **550** | — |

Además, este mes la salmonera firmó un **contrato de suministro**: AquaNutrí debe entregar **al menos $900$ t de Engorda**. Es una restricción de tipo $\ge$, y va a ser la protagonista.

$$\max\; z=530x_1+550x_2\qquad\text{s.a.}\quad
\begin{cases}
x_1+x_2\le 1.100 & \text{(E) extrusora}\\
0{,}5x_2\le 150 & \text{(H) harina de pescado}\\
0{,}8x_1+0{,}5x_2\le 900 & \text{(S) soya}\\
x_1\ \ge\ 900 & \text{(C) contrato}\\
x_1,x_2\ge0
\end{cases}$$

### Se pide

7. Resuelva el primal (gráficamente y/o con código) e identifique qué restricciones quedan **activas**.
8. Plantee el **dual**, cuidando el **signo** de cada variable dual según el tipo de su restricción primal.
9. Verifique **dualidad fuerte** ($z^*=w^*$) resolviendo el dual como un problema aparte.
10. Verifique las **holguras complementarias**, por restricción y por variable.
11. Calcule los precios sombra **a mano** y contrástelos con Gurobi. Interprete **cada uno** en términos de negocio, en particular el del contrato: ¿por qué es **negativo**?
12. La salmonera ofrece **subir el contrato a $950$ t** a cambio de un pago único de $\$800$ mil. ¿Conviene? ¿Puede responderlo con el precio sombra?

---

## 2.7 — El primal

_Resuelva el LP. Puede hacerlo **a mano** (el contrato y la extrusora acotan $x_2$; conviene despejar $z$ sobre la recta de la extrusora) y verificarlo con código. Identifique qué restricciones quedan **activas** y cuáles con **holgura**._

In [ ]:
# Resuelva el primal con gurobipy y reporte, por restriccion:
# tipo, RHS, lado izquierdo en el optimo, holgura, precio sombra (.Pi) y si esta activa.
#### CÓDIGO AQUÍ ####

In [ ]:
# Grafique la region factible con las cuatro restricciones y algunas curvas de nivel
# de la objetivo, marcando el optimo. Fijese en cuales rectas TOCAN la region factible.
#### CÓDIGO AQUÍ ####

## 2.8 a 2.11 — Dual, dualidad fuerte, holguras complementarias y precios sombra

_**(8)** Escriba el dual, cuidando el signo de cada $y_i$ según el tipo de su restricción primal (Sección 0.2)._

_**(9)** Verifique la dualidad fuerte: calcule $w^*=b^\top y^*$ y compárelo con $z^*$. Resuelva además el dual como un **problema aparte** y compruebe que su solución coincide con los `.Pi` del primal._

_**(10)** Verifique las holguras complementarias **por restricción** ($y_i\cdot\text{holgura}_i=0$) y **por variable** ($x_j\cdot\text{holgura dual}_j=0$)._

_**(11)** Calcule los precios sombra **a mano**: ¿qué restricciones duales están activas, y por qué? Resuelva el sistema resultante y contraste con Gurobi. Interprete **cada uno** en términos de negocio. Preste especial atención al del contrato: ¿por qué sale **negativo** y qué significa eso para la negociación con la salmonera?_

In [ ]:
# Plantee y resuelva el DUAL como un modelo aparte (ojo con el signo de la variable
# asociada a la restriccion >=), y verifique dualidad fuerte y holguras complementarias.
#### CÓDIGO AQUÍ ####

## 2.12 — ¿Conviene subir el contrato a 950 t? (pregunta 12)

_La salmonera ofrece un pago único de \$800 mil por comprometer $50$ t más de Engorda. Estime el efecto con el **precio sombra**, revise el **rango de validez** para saber si la estimación es legítima, y compruébelo **volviendo a resolver**. ¿Cuál es el precio de indiferencia? ¿Qué pasa si el contrato sube más allá de $1.100$ t?_

In [ ]:
# Reporte el rango de validez (.SARHSLow / .SARHSUp) del contrato y de la extrusora.
# Arme una tabla comparando la prediccion del precio sombra con el resultado de
# RE-RESOLVER, para varios valores del contrato. Cuidado: para valores grandes el
# problema puede volverse INFACTIBLE -- controle el Status del modelo.
#### CÓDIGO AQUÍ ####

---

# Ejercicio 3 — **KKT**: la teoría detrás de los precios sombra

Ahora vamos a mirar el mismo problema desde KKT. Primero verificamos que, para el LP del Ejercicio 2, **KKT no dice nada nuevo**: es la dualidad. Después rompemos la linealidad y vemos que **KKT sobrevive y la dualidad, tal como la conocíamos, no**.

### Contexto de la parte no lineal

El mercado del Smolt es **chico**: AquaNutrí es uno de tres proveedores en la región y, para colocar más toneladas, tiene que **bajar el precio**. El área comercial estima que el precio de venta cae linealmente con el volumen,

$$pr_{\text{SM}}(x_2)=1.600-0{,}04\,x_2\quad[\text{\$ mil/t}],$$

mientras el costo de la receta se mantiene en $\$1.050$ mil/t. El margen del Engorda no cambia (es un contrato a precio fijo). La función objetivo pasa a ser

$$f(x_1,x_2)=530\,x_1+\underbrace{\big(1.600-0{,}04x_2-1.050\big)}_{\text{margen unitario del Smolt}}x_2=530\,x_1+550\,x_2-0{,}04\,x_2^2,$$

**con las mismas cuatro restricciones del Ejercicio 2.**

### Se pide

**Parte (a) — KKT sobre el LP.**

13. Escriba el **Lagrangiano** del LP del Ejercicio 2 (todas las restricciones en forma $g_i(x)\le0$) y las **cuatro condiciones KKT**.
14. Verifique que $x^*=(900,200)$ con $\mu^*=(\mu_E,\mu_H,\mu_S,\mu_C)=(550,0,0,20)$ las satisface, y explique la relación entre $\mu^*$ y los precios sombra $y^*$ del Ejercicio 2. ¿Por qué $\mu_C=+20$ y $y_C=-20$?

**Parte (b) — KKT sobre el problema no lineal.**

15. Justifique que el problema es **cóncavo** y que, por lo tanto, KKT es **suficiente** y el punto que encuentre será óptimo **global**.
16. Escriba las condiciones KKT y resuélvalas **enumerando los casos** de restricciones activas/inactivas, como en la Clase 29. Reporte $x^*$, $f^*$ y $\mu^*$.
17. Verifique numéricamente que $\mu_i^*=\partial f^*/\partial b_i$ para la extrusora y para el contrato.
18. Compare con el LP del Ejercicio 2: ¿cambió el óptimo? ¿cambiaron los precios sombra? ¿qué le diría a la gerencia sobre el contrato ahora?

---

## Parte (a) — KKT sobre el LP (preguntas 13 y 14)

_**(13)** Lleve **todas** las restricciones a la forma $g_i(x)\le0$ —ojo con el contrato, que hay que dar vuelta—, escriba el Lagrangiano y las cuatro condiciones KKT._

_**(14)** Deduzca $\mu^*$ usando la holgura complementaria (¿qué multiplicadores se anulan de entrada?) y verifique las cuatro condiciones. Después explique la relación entre $\mu^*$ y los precios sombra $y^*$ del Ejercicio 2: ¿por qué el del contrato aparece como $+20$ en KKT y como $-20$ en Gurobi?_

In [ ]:
# Verifique numericamente las CUATRO condiciones KKT en x* = (900, 200):
#   1) estacionariedad     grad f = sum_i mu_i grad g_i
#   2) factibilidad primal g_i(x*) <= 0
#   3) factibilidad dual   mu_i >= 0
#   4) holgura complementaria  mu_i * g_i(x*) = 0
# y contraste mu con los precios sombra que reporto Gurobi en el Ejercicio 2.
#### CÓDIGO AQUÍ ####

## Parte (b) — KKT sobre el problema no lineal (preguntas 15 a 18)

_**(15)** Calcule $\nabla f$ y $\nabla^2 f$ y justifique que el problema es **cóncavo**. ¿Por qué eso convierte a KKT en condición **suficiente** de óptimo **global**? ¿Qué garantiza la **cualificación** de restricciones aquí?_

_**(16)** Escriba la estacionariedad y resuelva **enumerando los casos** de restricciones activas/inactivas, como en la Clase 29. Descarte cada caso indicando **por qué** (¿$\mu_i<0$? ¿$x_j<0$? ¿viola alguna restricción? ¿sistema incompatible?). Un descarte inicial ahorra la mitad del trabajo: mire la ecuación de $\partial f/\partial x_1$._

In [ ]:
# Defina f_nl (objetivo no lineal) y g_vec (las cuatro restricciones en forma g <= 0).
# Enumere los 2^4 subconjuntos de restricciones activas: para cada uno arme el sistema
#   estacionariedad (2 ecuaciones) + g_i(x) = 0 por cada activa
# resuelvalo, y marque el veredicto (KKT OK / mu<0 / x<0 / viola alguna / incompatible).
# Sugerencia: np.linalg.lstsq y revisar el residuo permite detectar los incompatibles.
#### CÓDIGO AQUÍ ####

In [ ]:
# Comprobacion independiente del optimo: scipy.optimize.minimize (SLSQP) y gurobipy
# (que admite objetivos cuadraticos). Deben coincidir con el caso KKT que sobrevivio.
#### CÓDIGO AQUÍ ####

### 3.17 — Los multiplicadores son los precios sombra (pregunta 17)

_Verifique que $\mu_i^*=\partial f^*/\partial b_i$ para la extrusora y para el contrato. Hágalo de las dos formas: **analíticamente** (escriba $f^*(b_E,b_C)$ suponiendo que el conjunto de activas no cambia, y derive) y **numéricamente** (diferencias finitas re-resolviendo el problema). ¿Por qué el numérico no da exacto?_

In [ ]:
# Escriba una funcion f_opt(bE, bC) que re-resuelva el problema no lineal, y estime
# df*/db por diferencias finitas centradas. Compare con mu_E = 534 y -mu_C = -4.
# Muestre ademas como cambia mu_E = 550 - 0.08*x2* al mover la capacidad.
#### CÓDIGO AQUÍ ####

In [ ]:
# Dos graficos:
#  (izq) la region factible con las curvas de nivel de f, y en x* los gradientes de las
#        restricciones ACTIVAS junto al gradiente de f. Sombree el CONO que generan:
#        la estacionariedad dice que grad f debe caer dentro de ese cono.
#  (der) el multiplicador de la extrusora como funcion de b_E, para el LP y para el
#        problema no lineal. ¿Cual es un escalon y cual una curva? ¿Por que?
#### CÓDIGO AQUÍ ####

_**(18)** Compare con el LP del Ejercicio 2 en una tabla: $x^*$, valor óptimo, restricciones activas, $\mu_E$ y $\mu_C$. ¿Se movió el óptimo? ¿Por qué? ¿Se movieron los precios? ¿Por qué la extrusora vale **menos** y el contrato cuesta **menos**?_

_Y la pregunta de negocio: con este modelo, ¿qué le responde ahora a la salmonera que ofrecía \$800 mil por 50 t más de contrato? ¿Cambió la recomendación respecto del Ejercicio 2, y por qué razón exactamente?_

---

## Resumen de la Ayudantía

### Modelamiento

| Estructura | Forma correcta | Tipo | Error típico |
|:--|:--|:--:|:--|
| Balance de masa | $p_i=\sum_k x_{ki}$ | $=$ | usar $\le$ y perder materia |
| Requerimiento de calidad | $\sum_k\pi_k x_{ki}\ge\pi^{\min}_i p_i$ | $\ge$ | escribirlo como cociente (no lineal, indefinido en $0$) |
| Disponibilidad / capacidad | $\sum_i x_{ki}\le S_k$ | $\le$ | olvidar el índice de periodo |
| Balance de inventario | $r_{it}=r_{i,t-1}+p_{it}-v_{it}$ | $=$ | omitir el **dato** $r^0_i$ del primer periodo |
| Lote mínimo + activación | $L^{\min}y_i\le p_i\le M y_i$ | binaria | modelar la disyunción sin binaria (imposible: no es convexa) |
| Compromiso y demanda | $\underline d_i\le v_i\le\bar d_i$ | cotas | escribirlas como restricciones (funciona, pero ensucia) |

### Dualidad y KKT

| Concepto | Idea clave |
|:--|:--|
| **Signo de la dual** | $\le\Rightarrow y_i\ge0$;  $\ge\Rightarrow y_i\le0$;  $=\Rightarrow y_i$ libre |
| **Dualidad fuerte** | $z^*=w^*$ en el óptimo |
| **Holguras complementarias** | $y_i(b_i-a_i^\top x^*)=0$ y $x_j^*(a_j^\top y^*-c_j)=0$ |
| **Precio sombra** | $y_i=\partial z^*/\partial b_i$; positivo = *cuánto pagaría por más*; **negativo = cuánto me cuesta que me obliguen a más** |
| **KKT** | estacionariedad + factibilidad primal + $\mu\ge0$ + holgura complementaria |
| **KKT $=$ dualidad, en un LP** | la estacionariedad **es** $A^\top y\ge c$, y $\mu^*=y^*$ (salvo signo si la restricción era $\ge$) |
| **Suficiencia** | problema cóncavo ($f$ cóncava, conjunto convexo) ⟹ todo punto KKT es óptimo **global** |
| **Resolución a mano** | enumerar activas/inactivas; descartar por $\mu_i<0$, por infactibilidad o por incompatibilidad |
| **En el caso no lineal** | $\mu_i=\partial f^*/\partial b_i$ **sigue valiendo**, pero es una **derivada**: cambia con el punto |
| **Un MIP no tiene duales** | hay que fijar la binaria en su óptimo (Ayudantía 7) |

**La idea para llevarse del curso.** Un modelo bien planteado entrega tres cosas, y la tercera es la más valiosa: un **plan** (qué hacer), un **valor** (cuánto se gana) y un **sistema de precios internos** (cuánto vale cada recurso escaso y cuánto cuesta cada obligación). Ese sistema de precios sale de la dualidad cuando el problema es lineal, de KKT cuando no lo es, y en ambos casos la regla que lo gobierna es la misma: **holgura complementaria**. Si un recurso sobra, no vale nada. Si vale algo, es porque se acabó.

---
*Ayudantía 8 — IIP314W Optimización Aplicada a Negocios — Universidad del Desarrollo — 2026-T2*